# Road Deterioration Dataset Checks

This notebook performs **read-only** checks on both datasets under `data/`: the Uganda Roads shapefile and the COCO road-surface image dataset. It uses Python's standard library, does not install packages, and does not modify source data.

Run cells from top to bottom. Review any reported mismatches before cleaning, transforming, or training.

In [4]:
from collections import Counter
from pathlib import Path
import json
import math
import struct
import zipfile

# Find the workspace root even if the notebook kernel starts in a subfolder.
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((path for path in candidates if (path / 'data').is_dir()), Path.cwd())
DATA_DIR = ROOT / 'data'
UGANDA_DIR = DATA_DIR / 'uganda_roads'
COCO_DIR = DATA_DIR / 'road_surface_condition_coco'
UGANDA_ZIP = DATA_DIR / 'uganda_roads.zip'
COCO_ZIP = DATA_DIR / 'road_surface_condition_coco.zip'
COCO_JSON = COCO_DIR / 'train' / '_annotations.coco.json'

print('Workspace:', ROOT)
print('Uganda Roads folder exists:', UGANDA_DIR.is_dir())
print('COCO dataset folder exists:', COCO_DIR.is_dir())

Workspace: c:\Users\HP\OneDrive\downloads\Desktop\ROAD-DETERIORATION
Uganda Roads folder exists: True
COCO dataset folder exists: True


## 1. Files and archive integrity

This checks the archive CRCs and verifies that the key shapefile and COCO manifest files are present. A full CRC pass reads every archive member and may take a little time.

In [6]:
def check_zip(path: Path) -> str:
    if not path.is_file():
        return 'NOT FOUND'
    with zipfile.ZipFile(path) as archive:
        bad_member = archive.testzip()
    return f'FAILED at {bad_member}' if bad_member else 'PASS'

print('Uganda ZIP CRC:', check_zip(UGANDA_ZIP))
print('COCO ZIP CRC:', check_zip(COCO_ZIP))

shapefile_components = ['Uganda_Roads.shp', 'Uganda_Roads.shx', 'Uganda_Roads.dbf', 'Uganda_Roads.prj']
print('Required shapefile components:')
for name in shapefile_components:
    print(f'  {name}: {"present" if (UGANDA_DIR / name).is_file() else "MISSING"}')
print('Optional text encoding file (.cpg):', 'present' if (UGANDA_DIR / 'Uganda_Roads.cpg').is_file() else 'missing')
print('COCO manifest:', 'present' if COCO_JSON.is_file() else 'MISSING')

Uganda ZIP CRC: PASS
COCO ZIP CRC: PASS
Required shapefile components:
  Uganda_Roads.shp: present
  Uganda_Roads.shx: present
  Uganda_Roads.dbf: present
  Uganda_Roads.prj: present
Optional text encoding file (.cpg): missing
COCO manifest: present


## 2. Uganda Roads shapefile header and attributes

The DBF header contains the attribute field names, storage types, widths, decimal precision, and row count. The geometry/index checks compare SHP records to SHX entries and DBF rows.

In [8]:
def read_dbf_header_and_rows(path: Path):
    raw = path.read_bytes()
    version = raw[0]
    record_count = struct.unpack_from('<I', raw, 4)[0]
    header_length, record_length = struct.unpack_from('<HH', raw, 8)
    fields = []
    offset = 32
    while offset < header_length and raw[offset] != 0x0D:
        descriptor = raw[offset:offset + 32]
        if len(descriptor) != 32:
            raise ValueError('Truncated DBF field descriptor')
        name = descriptor[:11].split(b'\0', 1)[0].decode('ascii', errors='replace')
        fields.append({'name': name, 'type': chr(descriptor[11]), 'width': descriptor[16], 'decimals': descriptor[17]})
        offset += 32
    blank_counts = Counter()
    for row_index in range(record_count):
        row = raw[header_length + row_index * record_length:header_length + (row_index + 1) * record_length]
        if len(row) != record_length:
            raise ValueError(f'Truncated DBF row {row_index + 1}')
        cursor = 1  # First byte is the dBASE deletion marker.
        for field in fields:
            value = row[cursor:cursor + field['width']].decode('cp1252', errors='replace').strip()
            if not value:
                blank_counts[field['name']] += 1
            cursor += field['width']
    return raw, version, record_count, header_length, record_length, fields, blank_counts

dbf, dbf_version, dbf_rows, dbf_header_len, dbf_record_len, dbf_fields, dbf_blanks = read_dbf_header_and_rows(UGANDA_DIR / 'Uganda_Roads.dbf')
print(f'DBF version: {dbf_version:#04x}; rows: {dbf_rows}; header bytes: {dbf_header_len}; record bytes: {dbf_record_len}')
print('Field headers:')
for field in dbf_fields:
    print(f"  {field['name']:<12} {field['type']} width={field['width']} decimals={field['decimals']}")
print('Blank fields:', dict(dbf_blanks) if dbf_blanks else 'none')
expected_dbf_bytes = dbf_header_len + dbf_rows * dbf_record_len
print('DBF expected bytes:', expected_dbf_bytes, '| actual:', len(dbf), '| trailing bytes:', len(dbf) - expected_dbf_bytes, '| final byte:', hex(dbf[-1]))

DBF version: 0x03; rows: 87; header bytes: 609; record bytes: 348
Field headers:
  LINKNO       N width=10 decimals=0
  ROADNO       C width=10 decimals=0
  START_KM     N width=10 decimals=0
  END_KM       N width=10 decimals=0
  LENGTH       N width=10 decimals=2
  STARTDESC    C width=50 decimals=0
  ENDDESC      C width=50 decimals=0
  CLASS        C width=20 decimals=0
  REGION       C width=30 decimals=0
  WIDTH        N width=10 decimals=2
  LANES        N width=5 decimals=0
  SURFTYPE     C width=30 decimals=0
  CONDITION    C width=30 decimals=0
  AADT         N width=10 decimals=0
  iso3         C width=50 decimals=0
  AICD_REG     N width=4 decimals=0
  TAH          N width=4 decimals=0
  Class_1      N width=4 decimals=0
Blank fields: {'ENDDESC': 6}
DBF expected bytes: 30885 | actual: 30886 | trailing bytes: 1 | final byte: 0x1a


In [9]:
def check_shapefile_bundle(directory: Path, dbf_count: int):
    shp = (directory / 'Uganda_Roads.shp').read_bytes()
    shx = (directory / 'Uganda_Roads.shx').read_bytes()
    header = {
        'file_code': struct.unpack_from('>i', shp, 0)[0],
        'declared_bytes': struct.unpack_from('>i', shp, 24)[0] * 2,
        'version': struct.unpack_from('<i', shp, 28)[0],
        'shape_type': struct.unpack_from('<i', shp, 32)[0],
        'bbox_xmin_ymin_xmax_ymax': struct.unpack_from('<4d', shp, 36),
    }
    records = []
    offset = 100
    while offset < len(shp):
        if offset + 8 > len(shp):
            raise ValueError('Truncated SHP record header')
        number, content_words = struct.unpack_from('>2i', shp, offset)
        end = offset + 8 + content_words * 2
        if end > len(shp):
            raise ValueError(f'Truncated SHP record {number}')
        shape_type = struct.unpack_from('<i', shp, offset + 8)[0]
        records.append((offset, content_words, shape_type))
        offset = end
    index_entries = [struct.unpack_from('>2i', shx, pos) for pos in range(100, len(shx), 8)]
    index_matches = len(index_entries) == len(records) and all(
        (index_entries[i][0] * 2, index_entries[i][1]) == records[i][:2]
        for i in range(len(records))
    )
    return header, records, index_entries, index_matches, dbf_count == len(records)

shp_header, shp_records, shx_entries, shx_matches, counts_match = check_shapefile_bundle(UGANDA_DIR, dbf_rows)
print('SHP header:', shp_header)
print('SHP geometry records:', len(shp_records), '| SHX entries:', len(shx_entries), '| DBF rows:', dbf_rows)
print('SHP record types:', dict(Counter(record[2] for record in shp_records)), '(3 = PolyLine)')
print('SHX offsets/lengths match SHP:', shx_matches)
print('SHP and DBF counts match:', counts_match)
print('SHP declared and actual bytes match:', shp_header['declared_bytes'] == (UGANDA_DIR / 'Uganda_Roads.shp').stat().st_size)

SHP header: {'file_code': 9994, 'declared_bytes': 445068, 'version': 1000, 'shape_type': 3, 'bbox_xmin_ymin_xmax_ymax': (29.717347936906233, -1.4242477683230372, 34.56693170318401, 3.5753544785666573)}
SHP geometry records: 87 | SHX entries: 87 | DBF rows: 87
SHP record types: {3: 87} (3 = PolyLine)
SHX offsets/lengths match SHP: True
SHP and DBF counts match: True
SHP declared and actual bytes match: True


## 3. COCO manifest, images, and segmentations

This validates image references and IDs, category usage, image dimensions, bounding boxes, and polygon segmentation coordinate structure. It reports issues rather than altering annotations.

In [11]:
coco = json.loads(COCO_JSON.read_text(encoding='utf-8-sig'))
images = coco.get('images', [])
annotations = coco.get('annotations', [])
categories = coco.get('categories', [])
category_names = {category['id']: category['name'] for category in categories}
image_by_id = {image['id']: image for image in images}
annotation_counts = Counter(annotation.get('category_id') for annotation in annotations)
missing_images = [image['file_name'] for image in images if not (COCO_DIR / 'train' / image['file_name']).is_file()]
duplicate_image_ids = len(image_by_id) != len(images)
annotation_image_ids_missing = [a.get('id') for a in annotations if a.get('image_id') not in image_by_id]
unknown_category_ids = [a.get('id') for a in annotations if a.get('category_id') not in category_names]
dimensions = Counter((image.get('width'), image.get('height')) for image in images)

print('Images:', len(images), '| dimensions:', dict(dimensions))
print('Annotations:', len(annotations))
print('Categories:', category_names)
print('Annotation counts by category:', {category_names.get(key, key): value for key, value in sorted(annotation_counts.items(), key=lambda item: str(item[0]))})
print('Missing referenced image files:', len(missing_images), missing_images[:5])
print('Duplicate image IDs:', duplicate_image_ids)
print('Annotations with unknown image IDs:', len(annotation_image_ids_missing))
print('Annotations with unknown category IDs:', len(unknown_category_ids))

Images: 3154 | dimensions: {(640, 640): 3154}
Annotations: 6304
Categories: {0: 'objects', 1: 'Cracks', 2: 'Damaged_Asphalt', 3: 'Open_Manhole', 4: 'Potholes', 5: 'Water_Filled_Potholes'}
Annotation counts by category: {'Cracks': 1740, 'Damaged_Asphalt': 1838, 'Open_Manhole': 126, 'Potholes': 1326, 'Water_Filled_Potholes': 1274}
Missing referenced image files: 0 []
Duplicate image IDs: False
Annotations with unknown image IDs: 0
Annotations with unknown category IDs: 0


In [10]:
bad_polygons = []
out_of_bounds_polygons = []
bad_bboxes = []
for annotation in annotations:
    image = image_by_id.get(annotation.get('image_id'))
    segmentation = annotation.get('segmentation')
    if not isinstance(segmentation, list) or not segmentation:
        bad_polygons.append(annotation.get('id'))
        continue
    for polygon in segmentation:
        if (not isinstance(polygon, list) or len(polygon) < 6 or len(polygon) % 2 != 0
                or not all(isinstance(value, (int, float)) and math.isfinite(value) for value in polygon)):
            bad_polygons.append(annotation.get('id'))
            break
        if image and any(x < 0 or x > image['width'] or y < 0 or y > image['height']
                         for x, y in zip(polygon[0::2], polygon[1::2])):
            out_of_bounds_polygons.append(annotation.get('id'))
            break
    bbox = annotation.get('bbox')
    if image and (not isinstance(bbox, list) or len(bbox) != 4
                  or not all(isinstance(value, (int, float)) and math.isfinite(value) for value in bbox)):
        bad_bboxes.append(annotation.get('id'))
    elif image:
        x, y, width, height = bbox
        if x < 0 or y < 0 or width < 0 or height < 0 or x + width > image['width'] + 1e-6 or y + height > image['height'] + 1e-6:
            bad_bboxes.append(annotation.get('id'))

print('Invalid/empty polygon records:', len(set(bad_polygons)))
print('Polygon coordinates outside image bounds:', len(set(out_of_bounds_polygons)))
print('Invalid/out-of-bounds bounding boxes:', len(set(bad_bboxes)))

split_directories = sorted(path.name for path in COCO_DIR.iterdir() if path.is_dir())
expected_splits = {'train', 'valid', 'validation', 'val', 'test'}
print('Dataset split directories found:', split_directories)
print('Expected split directories absent:', sorted(expected_splits - set(split_directories)))

Invalid/empty polygon records: 0
Polygon coordinates outside image bounds: 0
Invalid/out-of-bounds bounding boxes: 14
Dataset split directories found: ['train']
Expected split directories absent: ['test', 'val', 'valid', 'validation']


In [13]:
# Inspect each bounding-box issue; do not clip or change source annotations.
bbox_issues = []
for annotation in annotations:
    image = image_by_id.get(annotation.get('image_id'))
    bbox = annotation.get('bbox')
    if image is None or not isinstance(bbox, list) or len(bbox) != 4:
        continue
    x, y, width, height = bbox
    right_excess = max(0.0, x + width - image['width'])
    bottom_excess = max(0.0, y + height - image['height'])
    if x < 0 or y < 0 or width < 0 or height < 0 or right_excess > 1e-6 or bottom_excess > 1e-6:
        bbox_issues.append({
            'annotation_id': annotation.get('id'),
            'image': image['file_name'],
            'bbox_xywh': bbox,
            'right_excess_px': round(right_excess, 6),
            'bottom_excess_px': round(bottom_excess, 6),
        })

print(f'Bounding-box issues: {len(bbox_issues)}')
for issue in bbox_issues:
    print(issue)
if bbox_issues:
    print('Review these annotations; this diagnostic does not modify or clip any boxes.')

Bounding-box issues: 14
{'annotation_id': 635, 'image': 'IMG_20260501_203247_jpg.rf.49ebba2b820c83e7031d83dfe32435e0.jpg', 'bbox_xywh': [197.13, 69.53, 442.88, 391.72], 'right_excess_px': 0.01, 'bottom_excess_px': 0.0}
{'annotation_id': 2446, 'image': 'IMG_20260501_183559_jpg.rf.34f49e6b917ed7e4c1daf030cfec4427.jpg', 'bbox_xywh': [506.88, 343.91, 133.13, 93.13], 'right_excess_px': 0.01, 'bottom_excess_px': 0.0}
{'annotation_id': 2649, 'image': 'IMG_20260501_171008_jpg.rf.f1038d285a597c78bea013bc5f0d22a8.jpg', 'bbox_xywh': [371.88, 298.91, 268.13, 150.31], 'right_excess_px': 0.01, 'bottom_excess_px': 0.0}
{'annotation_id': 2707, 'image': 'IMG_20260501_181428_jpg.rf.af8eca34aad46222f3f7157392f215b9.jpg', 'bbox_xywh': [102.63, 493.59, 537.38, 59.06], 'right_excess_px': 0.01, 'bottom_excess_px': 0.0}
{'annotation_id': 2759, 'image': 'IMG_20260501_172532_jpg.rf.a22336d7576fc1d043649390057b0d3a.jpg', 'bbox_xywh': [574.63, 124.02, 65.38, 148.32], 'right_excess_px': 0.01, 'bottom_excess_px': 0

## 4. Review checklist

Compare the outputs with the current audit notes. The COCO archive currently contains a training split only; it has no validation/test data. The Uganda Roads metadata reports 281 features although its SHP/SHX/DBF files contain 87. These are review items, not changes made by this notebook.

Keep the original archives as immutable source data. Decide how to handle missing values, unknown labels, and train/validation/test splitting before modeling.